# Orchestrator + UseModel Integration (Minimal Change)

This notebook combines only two existing workflows:

1. `Orchestrator_Pipeline.ipynb` for routing
2. `usemodelkaggle.ipynb` for model loading + inference

Target flow:
- User prompt
- Orchestrator routes to `script reviewer`.
- Impact Agent is not working in this orchestrator, but the router can tell it, and stop the running session in gradio
- UseModel pipeline runs and returns one final output

Running this Orchestrator:
- Open it in your VS Code.
- In the Extensions at the left bar, find `Colab` from `Google.com` (current install count is around 225,000), and install it.
- After installing, click the `Select Kernal` at the top right, then `Select Another Kernal`, and follow the sequence below:
  - `Colab`
  - `New Colab Server`
  - `GPU`
  - `A100` should be good
  - `High-RAM`
  - Give it a name, less than 10 characters. I used `A100-high`
  - Wait for a few seconds, then you can select the programming language (just like what you did in browser). Select `Python3 (ipykernel)`. 
  - START SURF!


In [1]:
# ============================================================
# STEP 0: Progress Checkpoint
# ============================================================
print("[STEP 0] Notebook loaded.")
print("[STEP 0.1] Next: install dependencies.")


[STEP 0] Notebook loaded.
[STEP 0.1] Next: install dependencies.


In [2]:
# ============================================================
# STEP 1: Install Dependencies (copy from existing notebooks)
# ============================================================
print("[STEP 1] Installing dependencies...")

!pip -q install -U google-generativeai gradio
!pip -q install -U google-api-python-client google-auth-httplib2 google-auth-oauthlib
!pip -q install -U unsloth unsloth-zoo modelscope peft accelerate bitsandbytes huggingface_hub python-docx PyPDF2
!pip -q install -U "sentence-transformers>=3.0.1" "faiss-cpu>=1.8.0" "langchain-text-splitters>=0.3.0" "pypdf>=4.2.0" "transformers>=4.45.0"

print("[STEP 1.1] Install complete.")


[STEP 1] Installing dependencies...
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
unsloth-zoo 2026.2.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.4,!=4.57.5,<=4.57.6,>=4.51.3, but you have transformers 5.2.0 which is incompatible.
unsloth 2026.2.1 requires transformers!=4.52.0,!=4.52.1,!=4.52.2,!=4.52.3,!=4.53.0,!=4.54.0,!=4.55.0,!=4.55.1,!=4.57.0,!=4.57.4,!=4.57.5,<=4.57.6,>=4.51.3, but you have transformers 5.2.0 which is incompatible.
[STEP 1.1] Install complete.


In [3]:
# ============================================================
# STEP 2: Imports + Runtime Flags
# ============================================================
print("[STEP 2] Importing libraries...")

import os
import re
import torch
import getpass
import gradio as gr
from threading import Lock
from types import SimpleNamespace

import google.generativeai as genai
from unsloth import FastLanguageModel
from peft import PeftModel

os.environ["UNSLOTH_DISABLE_LOG_STATS"] = "1"
os.environ.setdefault("UNSLOTH_USE_MODELSCOPE", "1")

print("[STEP 2.1] Imports ready.")


[STEP 2] Importing libraries...


/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
[STEP 2.1] Imports ready.


In [ ]:
# ============================================================
# STEP 3: Keys / Tokens
# ============================================================
print("[STEP 3] Loading credentials...")

if not os.getenv("GOOGLE_API_KEY"):
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Enter GOOGLE_API_KEY: ")
if not os.getenv("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass.getpass("Enter HF_TOKEN: ")


print("[STEP 3.1] Credentials loaded.")


[STEP 3] Loading credentials...
[STEP 3.1] Credentials loaded.


In [5]:
# ============================================================
# STEP 4: RequestOrchestrator (copied from Orchestrator_Pipeline)
# ============================================================
print("[STEP 4] Defining orchestrator class...")

class RequestOrchestrator:
    """
    Orchestrator that determines which agent should handle a user's request.
    Routes between Impact Agent and Script Reviewer.
    """

    IMPACT_AGENT = "impact agent"
    SCRIPT_REVIEWER = "script reviewer"

    def __init__(self, api_key: str, model_name: str = "gemini-2.5-flash"):
        genai.configure(api_key=api_key)
        self.model = genai.GenerativeModel(model_name)
        print(f"[STEP 4.1] Orchestrator initialized with {model_name}")

    def _create_routing_prompt(self, user_prompt: str) -> str:
        system_prompt = f"""You are a routing system that determines which agent should handle a user's request.

You have TWO agents available:

1. **Impact Agent**: Handles questions about:
   - Impact, outcomes, effects, consequences, results
   - Benefits, changes, influence of actions/policies/programs
   - Analysis of societal, environmental, or business impacts
   - "What if" scenarios and their implications

2. **Script Reviewer**: Handles questions about:
   - Reviewing, analyzing, or critiquing scripts (movie, TV, theater)
   - Code review and programming help
   - Document review and editing
   - Writing feedback and improvements

USER QUERY: {user_prompt}

Based on the user's query, determine which agent should handle this request.

RESPOND WITH ONLY ONE OF THESE TWO OPTIONS (exactly as written):
- "impact agent"
- "script reviewer"

Your response should contain ONLY the agent name, nothing else."""
        return system_prompt

    def route_request(self, user_prompt: str) -> str:
        prompt = self._create_routing_prompt(user_prompt)
        try:
            response = self.model.generate_content(prompt)
            decision = response.text.strip().lower()
            if self.IMPACT_AGENT in decision:
                return self.IMPACT_AGENT
            elif self.SCRIPT_REVIEWER in decision:
                return self.SCRIPT_REVIEWER
            else:
                print(f"[STEP 4 WARN] Unexpected routing response: {decision}. Using fallback.")
                return self._fallback_routing(user_prompt)
        except Exception as e:
            print(f"[STEP 4 WARN] Routing error: {str(e)}. Using fallback.")
            return self._fallback_routing(user_prompt)

    def _fallback_routing(self, user_prompt: str) -> str:
        user_lower = user_prompt.lower()

        script_keywords = ["script", "review", "code", "analyze", "critique",
                          "feedback", "document", "written", "check", "screenplay",
                          "program", "function", "bug", "error", "syntax"]

        impact_keywords = ["impact", "effect", "outcome", "result", "consequence",
                          "benefit", "change", "influence", "affect", "what if",
                          "implications", "happens", "caused by"]

        script_score = sum(1 for keyword in script_keywords if keyword in user_lower)
        impact_score = sum(1 for keyword in impact_keywords if keyword in user_lower)

        if script_score > impact_score:
            return self.SCRIPT_REVIEWER
        else:
            return self.IMPACT_AGENT

print("[STEP 4.2] Orchestrator class ready.")


[STEP 4] Defining orchestrator class...
[STEP 4.2] Orchestrator class ready.


In [6]:
# ============================================================
# STEP 5: Default User Prompt (for UI prefill)
# ============================================================
print("[STEP 5] Setting default prompt/context for Gradio UI...")

# This default text is prefilled in the input box.
# You can edit it in UI before clicking Analyze.
USER_PROMPT = "Please review my script opening and tell me what to improve."
ADDITIONAL_CONTEXT = "Focus on clarity, pacing, and emotional impact."

# Keep for backward compatibility with STEP 10 one-shot test.
# In Gradio flow, users upload files directly (no Drive file path needed here).
UPLOADED_FILE_PATH = None

print("[STEP 5.1] Default USER_PROMPT set.")
print("[STEP 5.2] Default CONTEXT set.")
print("[STEP 5.3] File will come from Gradio upload.")


[STEP 5] Setting default prompt/context for Gradio UI...
[STEP 5.1] Default USER_PROMPT set.
[STEP 5.2] Default CONTEXT set.
[STEP 5.3] File will come from Gradio upload.


In [7]:
# ============================================================
# OPTIONAL: Gemini Connectivity Quick Check
# ============================================================
print("[OPTIONAL] Skip this cell if routing is already working.")
print("[OPTIONAL] This cell is intentionally lightweight to avoid long hangs.")


[OPTIONAL] Skip this cell if routing is already working.
[OPTIONAL] This cell is intentionally lightweight to avoid long hangs.


## Legacy Step (Skip)
This old Step 6 cell is deprecated.
Use the next code cell: **STEP 6: Route Prompt with Orchestrator (API timeout + fallback)**.


In [8]:
# ============================================================
# STEP 6: Initialize Router (routing executes only on Analyze click)
# ============================================================
print("[STEP 6] Initializing orchestrator and safe_route...")

ROUTER_MODEL = os.getenv("ROUTER_MODEL", "gemini-2.5-flash-lite")
orchestrator = RequestOrchestrator(api_key=os.environ["GOOGLE_API_KEY"], model_name=ROUTER_MODEL)
print(f"[STEP 6.0] Router model: {ROUTER_MODEL}")


def keyword_route(prompt: str):
    text = (prompt or "").lower()

    script_keywords = [
        "script", "review", "screenplay", "dialogue", "plot", "character",
        "code", "bug", "function", "error", "syntax", "refactor", "document"
    ]
    impact_keywords = [
        "impact", "outcome", "consequence", "benefit", "harm", "society",
        "community", "ethics", "policy", "what if", "implication"
    ]

    script_score = sum(1 for k in script_keywords if k in text)
    impact_score = sum(1 for k in impact_keywords if k in text)

    if script_score > 0 and impact_score == 0:
        return orchestrator.SCRIPT_REVIEWER, script_score, impact_score
    if impact_score > 0 and script_score == 0:
        return orchestrator.IMPACT_AGENT, script_score, impact_score

    return None, script_score, impact_score


def safe_route(prompt: str, timeout_sec: int = 12):
    # 1) Fast keyword route first (always prompt-based)
    kw_agent, s_score, i_score = keyword_route(prompt)
    if kw_agent is not None:
        return kw_agent, f"keyword(script={s_score},impact={i_score})"

    # 2) LLM route only when keyword route is ambiguous
    try:
        routing_prompt = orchestrator._create_routing_prompt(prompt)
        response = orchestrator.model.generate_content(
            routing_prompt,
            request_options={"timeout": timeout_sec},
        )
        decision = (response.text or "").strip().lower()

        if orchestrator.IMPACT_AGENT in decision:
            return orchestrator.IMPACT_AGENT, "llm"
        if orchestrator.SCRIPT_REVIEWER in decision:
            return orchestrator.SCRIPT_REVIEWER, "llm"

        fb = orchestrator._fallback_routing(prompt)
        return fb, f"fallback_format(script={s_score},impact={i_score})"

    except Exception as e:
        fb = orchestrator._fallback_routing(prompt)
        return fb, f"fallback_error:{type(e).__name__}(script={s_score},impact={i_score})"


print("[STEP 6.1] safe_route is ready.")
print("[STEP 6.2] Routing runs in STEP 11 when Analyze is clicked.")


[STEP 6] Initializing orchestrator and safe_route...
[STEP 4.1] Orchestrator initialized with gemini-2.5-flash-lite
[STEP 6.0] Router model: gemini-2.5-flash-lite
[STEP 6.1] safe_route is ready.
[STEP 6.2] Routing runs in STEP 11 when Analyze is clicked.


In [9]:
# ============================================================
# STEP 7: Mount Google Drive + set adapter path
# ============================================================
print("[STEP 7] Mount Drive and configure adapter path...")

import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[STEP 7.1] Drive mounted (or already mounted).")
except Exception as e:
    print("[STEP 7 WARN] Could not mount Drive in this runtime:", e)
finally:
    print("""
          ####################################################################################################################
          NOTE: You need to mount your Google Drive to access the the model files. 
          Before mounting, ensure that you have uploaded your model file to your Drive. 
          If you don't have the model files in Drive, it's located at https://drive.google.com/drive/folders/1uN9ZmE1PFcXRQQik_VAfoHFY1llRwO2I?usp=drive_link.
          ________________________________________________________________________________________________
          I put the model files in my Drive's 1st level (not in a subfolder) for easier access. 
          So after mounting, please modify the ADAPTER_DIR path below if your files are in a different location or subfolder.
          ####################################################################################################################
          """)

[STEP 7] Mount Drive and configure adapter path...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[STEP 7.1] Drive mounted (or already mounted).

          ####################################################################################################################
          NOTE: You need to mount your Google Drive to access the the model files. 
          Before mounting, ensure that you have uploaded your model file to your Drive. 
          If you don't have the model files in Drive, it's located at https://drive.google.com/drive/folders/1uN9ZmE1PFcXRQQik_VAfoHFY1llRwO2I?usp=drive_link.
          ________________________________________________________________________________________________
          I put the model files in my Drive's 1st level (not in a subfolder) for easier access. 
          So after mounting, please modify the ADAPTER_DIR path below if your files are in a different locati

In [10]:
ADAPTER_DIR = "/content/drive/MyDrive/my_finetuned_llm"   # Change this if your files are in a different location or subfolder in Drive.

print("[STEP 7.2] ADAPTER_DIR:", ADAPTER_DIR)
print("adapter exists =", os.path.exists(ADAPTER_DIR))
print("config exists =", os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")))

if not os.path.exists(os.path.join(ADAPTER_DIR, "adapter_config.json")):
    raise FileNotFoundError(
        f"adapter_config.json not found in {ADAPTER_DIR}. "
        "Please verify Drive mount and folder location."
    )

print("[STEP 7.3] Files in ADAPTER_DIR:")
for f in sorted(os.listdir(ADAPTER_DIR)):
    print(f"  {f}")

[STEP 7.2] ADAPTER_DIR: /content/drive/MyDrive/my_finetuned_llm
adapter exists = True
config exists = True
[STEP 7.3] Files in ADAPTER_DIR:
  README.md
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  special_tokens_map.json
  tokenizer.json
  tokenizer_config.json


In [11]:
# ============================================================
# STEP 8: Load Base Model + LoRA (copied from usemodelkaggle)
# ============================================================
print("[STEP 8] Loading fine-tuned model stack...")

BASE_ID = "unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit"
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    raise ValueError("Missing HF_TOKEN. Set it before running this cell.")

try:
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN,
        disable_log_stats=True,
    )
except TimeoutError:
    os.environ["UNSLOTH_USE_MODELSCOPE"] = "1"
    model, tokenizer = FastLanguageModel.from_pretrained(
        BASE_ID,
        max_seq_length=2048,
        load_in_4bit=True,
        token=HF_TOKEN,
        disable_log_stats=True,
    )

model = PeftModel.from_pretrained(model, ADAPTER_DIR)
FastLanguageModel.for_inference(model)
model.eval()

print("[STEP 8.1] Model + LoRA ready.")


[STEP 8] Loading fine-tuned model stack...
==((====))==  Unsloth 2026.2.1: Fast Llama patching. Transformers: 5.2.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

Unsloth: Will load unsloth/meta-llama-3.1-8b-instruct-unsloth-bnb-4bit as a legacy tokenizer.


[STEP 8.1] Model + LoRA ready.


In [12]:
# ============================================================
# STEP 9: Inference Function (copied from usemodelkaggle, non-Gradio)
# ============================================================
print("[STEP 9] Defining evaluator...")

import io

tok = tokenizer
mdl = model
mdl.eval()

device = "cuda" if torch.cuda.is_available() else "cpu"
gen_lock = Lock()
history = []

_DRIVE_API = None


def _init_drive_api():
    global _DRIVE_API
    if _DRIVE_API is not None:
        return _DRIVE_API

    try:
        from google.colab import auth
        auth.authenticate_user()
    except Exception:
        # Non-Colab env may still have ADC credentials.
        pass

    try:
        import google.auth
        from googleapiclient.discovery import build
        creds, _ = google.auth.default(scopes=["https://www.googleapis.com/auth/drive.readonly"])
        _DRIVE_API = build("drive", "v3", credentials=creds, cache_discovery=False)
        return _DRIVE_API
    except Exception as e:
        print(f"[STEP 9 WARN] Drive API init failed: {e}")
        return None


def _extract_drive_id_from_url(url: str) -> str:
    if not url:
        return ""
    patterns = [
        r"/d/e/([a-zA-Z0-9_-]+)",
        r"/d/([a-zA-Z0-9_-]+)",
        r"[?&]id=([a-zA-Z0-9_-]+)",
    ]
    for pat in patterns:
        m = re.search(pat, url)
        if m:
            return m.group(1)
    return ""


def _read_google_shortcut_meta(path: str):
    """
    Read .gdoc/.gsheet/.gslides shortcut metadata.
    Supports JSON shortcut files and INI-style URL shortcuts.
    Returns (file_id, url).
    """
    url = ""
    file_id = ""

    raw = ""
    try:
        with open(path, "r", encoding="utf-8", errors="ignore") as f:
            raw = f.read()
    except Exception:
        return "", ""

    # Try JSON first
    try:
        meta = json.loads(raw)
        if isinstance(meta, dict):
            url = str(meta.get("url", "") or "")
            file_id = str(meta.get("doc_id", "") or "")

            resource_id = str(meta.get("resource_id", "") or "")
            if not file_id and resource_id:
                # examples: "document/FILE_ID" or "document:FILE_ID"
                if "/" in resource_id:
                    file_id = resource_id.split("/")[-1]
                elif ":" in resource_id:
                    file_id = resource_id.split(":")[-1]

            if not file_id:
                file_id = _extract_drive_id_from_url(url)

            if file_id:
                return file_id, url
    except Exception:
        pass

    # Fallback for non-JSON shortcut content
    # e.g. [InternetShortcut] URL=https://docs.google.com/document/d/...
    m = re.search(r"(?im)^URL\s*=\s*(https?://\S+)", raw)
    if m:
        url = m.group(1).strip()
    else:
        m2 = re.search(r"https?://\S+", raw)
        if m2:
            url = m2.group(0).strip()

    if not file_id and url:
        file_id = _extract_drive_id_from_url(url)

    return file_id, url


def _google_mime_for_shortcut_ext(ext: str) -> str:
    ext = ext.lower()
    mapping = {
        ".gdoc": "application/vnd.google-apps.document",
        ".gsheet": "application/vnd.google-apps.spreadsheet",
        ".gslides": "application/vnd.google-apps.presentation",
        ".gdraw": "application/vnd.google-apps.drawing",
        ".gform": "application/vnd.google-apps.form",
        ".gsite": "application/vnd.google-apps.site",
    }
    return mapping.get(ext, "")


def _lookup_google_file_id_by_name(local_path: str):
    """
    Fallback when .g* shortcut metadata has no file id.
    Lookup by base filename in user's Drive.
    Returns (file_id, webViewLink).
    """
    svc = _init_drive_api()
    if svc is None:
        return "", ""

    stem = os.path.splitext(os.path.basename(local_path))[0]
    ext = os.path.splitext(local_path)[1].lower()
    mime = _google_mime_for_shortcut_ext(ext)

    safe_name = stem.replace("'", "\'")
    q = f"name = '{safe_name}' and trashed = false"
    if mime:
        q += f" and mimeType = '{mime}'"

    try:
        resp = svc.files().list(
            q=q,
            corpora="user",
            pageSize=10,
            fields="files(id,name,mimeType,webViewLink)",
        ).execute()
        files = resp.get("files", [])
        if files:
            return files[0].get("id", ""), files[0].get("webViewLink", "")
    except Exception:
        pass

    # broad fallback: drop mime filter
    try:
        resp = svc.files().list(
            q=f"name = '{safe_name}' and trashed = false",
            corpora="user",
            pageSize=10,
            fields="files(id,name,mimeType,webViewLink)",
        ).execute()
        files = resp.get("files", [])
        if files:
            return files[0].get("id", ""), files[0].get("webViewLink", "")
    except Exception:
        pass

    return "", ""


def _preferred_export_mimes(ext: str):
    ext = ext.lower()
    if ext == ".gsheet":
        return ["text/csv", "application/pdf", "text/plain"]
    if ext == ".gslides":
        return ["text/plain", "application/pdf"]
    if ext == ".gdraw":
        return ["application/pdf", "image/svg+xml", "text/plain"]
    if ext == ".gdoc":
        return ["text/plain", "application/pdf"]
    # generic .g* fallback
    return ["text/plain", "application/pdf", "text/csv"]


def _decode_export_bytes(data, mime_type: str):
    if isinstance(data, str):
        return data

    if mime_type == "application/pdf":
        import PyPDF2
        reader = PyPDF2.PdfReader(io.BytesIO(data))
        return "\n".join([page.extract_text() or "" for page in reader.pages])

    for enc in ("utf-8", "latin-1"):
        try:
            return data.decode(enc, errors="ignore")
        except Exception:
            pass
    return str(data)


def read_google_workspace_file(path: str) -> str:
    ext = os.path.splitext(path)[1].lower()
    if not ext.startswith('.g'):
        return ""

    file_id, web_url = _read_google_shortcut_meta(path)

    # fallback by Drive lookup if shortcut meta didn't include file id
    if not file_id:
        looked_id, looked_url = _lookup_google_file_id_by_name(path)
        if looked_id:
            file_id = looked_id
            if not web_url:
                web_url = looked_url

    if not file_id:
        return f"(Google shortcut found but file id not found: {path})"

    svc = _init_drive_api()
    if svc is None:
        return f"(Drive API not available for reading {path})"

    last_err = None
    for mime in _preferred_export_mimes(ext):
        try:
            payload = svc.files().export(fileId=file_id, mimeType=mime).execute()
            text = _decode_export_bytes(payload, mime)
            if text and text.strip():
                return text
        except Exception as e:
            last_err = e

    # metadata fallback
    try:
        meta = svc.files().get(fileId=file_id, fields="name,mimeType,webViewLink").execute()
        return (
            f"(Could not export content. name={meta.get('name')} mime={meta.get('mimeType')} "
            f"link={meta.get('webViewLink', web_url)})"
        )
    except Exception:
        return f"(Could not export Google file content. Last error: {last_err})"


def format_output(raw):
    text = raw.replace("####", "").replace("**", "")
    sections = {
        "verdict": r"(?i)verdict[:\-]\s*(.*?)(?=\n|score|benefits|$)",
        "score": r"(?i)score[:\-]\s*([0-9\.]+)",
        "benefits": r"(?i)benefits?[:\-]\s*(.*?)(?=\n\s*risks?|\n\s*overall|$)",
        "risks": r"(?i)risks?[:\-]\s*(.*?)(?=\n\s*overall|\n\s*rationale|$)",
        "overall": r"(?i)(?:overall|rationale|analysis)[:\-]\s*(.*)",
    }
    formatted = {}
    for key, pattern in sections.items():
        m = re.search(pattern, text, re.S)
        formatted[key] = m.group(1).strip() if m else ""
    return formatted




def evaluate_story(story, context, uploaded_file):
    with gen_lock:
        if (not story or story.strip() == "") and not uploaded_file:
            return "Please enter a story or upload a file.", 1, history

        file_text = ""
        if uploaded_file is not None:
            path = uploaded_file.name
            ext = os.path.splitext(path)[1].lower()
            try:
                if ext == ".txt":
                    with open(path, "r", encoding="utf-8", errors="ignore") as f:
                        file_text = f.read()
                elif ext in (".docx", ".doc"):
                    import docx
                    doc = docx.Document(path)
                    file_text = "\n".join([p.text for p in doc.paragraphs])
                elif ext == ".pdf":
                    import PyPDF2
                    with open(path, "rb") as f:
                        reader = PyPDF2.PdfReader(f)
                        file_text = "\n".join([page.extract_text() or "" for page in reader.pages])
                elif ext.startswith('.g'):
                    file_text = read_google_workspace_file(path)
                else:
                    file_text = "(File type not supported)"
            except Exception as e:
                file_text = f"(Error reading file: {str(e)})"

        final_story = (story or "") + "\n\n" + file_text
        if final_story.strip() == "":
            return "Input is empty.", 1, history

        print("\n" + "=" * 100)
        print("[DEBUG] FINAL STORY (user prompt + uploaded script text)")
        print("=" * 100)
        print(final_story)
        print("=" * 100 + "\n")

        template = """Below is an instruction that describes a task.

### Instruction:
Evaluate whether the following idea uplifts humanity. Provide:
- Verdict (Yes/No)
- Score (1 to 5)
- Benefits
- Risks
- Overall rationale
Be objective.

### Input:
{story}

### Additional Context:
{context}

### Response:
"""

        prompt = template.format(
            story=final_story.strip(),
            context=context.strip() if context else "None",
        )

        print("\n" + "=" * 100)
        print("[DEBUG] FULL PROMPT SENT TO MODEL")
        print("=" * 100)
        print(prompt)
        print("=" * 100 + "\n")

        inputs = tok(prompt, return_tensors="pt").to(device)
        outputs = mdl.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.4,
            top_p=0.95,
            do_sample=True,
        )

        prompt_len = inputs["input_ids"].shape[1]
        raw_text = tok.decode(outputs[0][prompt_len:], skip_special_tokens=True).strip()
        parsed = format_output(raw_text)

        try:
            val = float(parsed["score"])
            final_score = int(round(val))
            final_score = max(1, min(5, final_score))
        except Exception:
            final_score = 1

        history.append({
            "story": (final_story[:200] + "...") if len(final_story) > 200 else final_story,
            "score": final_score,
            "verdict": parsed["verdict"],
            "output": (raw_text[:300] + "...") if len(raw_text) > 300 else raw_text,
        })

        display_text = f"""
### Verdict
{parsed['verdict']}

### Score
{parsed['score'] if parsed['score'] else final_score}

### Benefits
{parsed['benefits']}

### Risks
{parsed['risks']}

### Overall Rationale
{parsed['overall']}
""".strip()

        return display_text, final_score, history

print("[STEP 9.1] Evaluator ready.")


[STEP 9] Defining evaluator...
[STEP 9.1] Evaluator ready.


## STEP 13: Impact Agent Backend (Copied from Impact_RAG_v4)

This section is copied from `Impact_RAG_v4.ipynb` with minimal modifications.
It defines the RAG-backed impact analysis path used by the unified dispatcher.


In [13]:
# === Choose your generator backend ===
# "local" uses a small open-source chat model via Hugging Face Transformers.
# "openai" uses OpenAI's API (set OPENAI_API_KEY).
# "gemini" uses Google Generative AI (set GOOGLE_API_KEY).
GEN_BACKEND = "gemini"  # options: "local", "openai", "gemini"

# Local model config:
LOCAL_MODEL = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # light model for CPU/GPU demos

# Retrieval config
EMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHUNK_SIZE = 800
CHUNK_OVERLAP = 120
TOP_K = 4

# Prompt template
SYSTEM_PROMPT = (
    "You are a helpful assistant. Use the provided context to answer the user's question.\n"
    "If the answer is not in the context, say you don't know.\n"
)

ANSWER_TEMPLATE = """[System]
{system}

[Context]
{context}

[User Question]
{question}

[Instructions]
- Cite the most relevant chunks briefly (e.g., 'From chunk 2').
- If unsure, say 'I don't know from the provided docs.'
- Keep answers concise and factual.
"""

OPENAI_API_KEY = globals().get("OPENAI_API_KEY", os.getenv("OPENAI_API_KEY", ""))
GOOGLE_API_KEY = globals().get("GOOGLE_API_KEY", os.getenv("GOOGLE_API_KEY", ""))

In [14]:
# Copied runtime imports from Impact_RAG_v4
import os, torch, faiss
from sentence_transformers import SentenceTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

print("Impact backend runtime | PyTorch:", torch.__version__, "| CUDA:", torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"


Impact backend runtime | PyTorch: 2.10.0+cu128 | CUDA: True


In [15]:
import os
from typing import List, Dict
from pypdf import PdfReader
from docx import Document  # ADDED THIS LINE

def load_texts_from_paths(paths: List[str]) -> List[Dict]:
    """
    Load text from various file formats: PDF, DOCX, DOC, TXT, MD
    """
    docs = []
    for p in paths:
        text = ""

        # PDF files
        if p.lower().endswith(".pdf"):
            try:
                reader = PdfReader(p)
                for page in reader.pages:
                    text += page.extract_text() or ""
                print(f"✓ Loaded PDF: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse PDF {p}: {e}")
                continue

        # DOCX files (Word documents) # NEW
        elif p.lower().endswith(".docx"):
            try:
                doc = Document(p)
                # Extract text from all paragraphs
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                # Also extract text from tables
                for table in doc.tables:
                    for row in table.rows:
                        for cell in row.cells:
                            text += "\n" + cell.text
                print(f"✓ Loaded DOCX: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOCX {p}: {e}")
                continue

        # DOC files (legacy Word format) # NEW
        elif p.lower().endswith(".doc"):
            try:
                doc = Document(p)
                text = "\n".join([paragraph.text for paragraph in doc.paragraphs])
                print(f"✓ Loaded DOC: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to parse DOC {p}: {e}")
                print("     Note: Legacy .doc files may require conversion to .docx")
                continue

        # TXT and MD files
        elif p.lower().endswith((".txt", ".md")):
            try:
                with open(p, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                print(f"✓ Loaded TXT/MD: {os.path.basename(p)} ({len(text)} chars)")
            except Exception as e:
                print(f"[WARN] Failed to read text file {p}: {e}")
                continue

        else:
            print(f"[SKIP] Unsupported file type: {p}")
            continue

        if text.strip():  # Only add if we got some text
            docs.append({"path": p, "text": text})
        else:
            print(f"[WARN] No text extracted from {p}")

    return docs

splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP, length_function=len
)

def chunk_docs(docs: List[Dict]) -> List[Dict]:
    chunks = []
    for d in docs:
        for i, ch in enumerate(splitter.split_text(d["text"])):
            chunks.append({"source": d["path"], "chunk_id": i, "text": ch})
    return chunks

class RAGIndex:
    def __init__(self, embedding_model_name: str):
        self.model = SentenceTransformer(embedding_model_name, device=device)
        self.index = None
        self.chunks: List[Dict] = []

    def build(self, chunks: List[Dict]):
        self.chunks = chunks
        embs = self.model.encode([c["text"] for c in chunks], convert_to_numpy=True, show_progress_bar=True)
        dim = embs.shape[1]
        index = faiss.IndexFlatIP(dim)
        faiss.normalize_L2(embs)
        index.add(embs)
        self.index = index
        print(f"Built index with {len(chunks)} chunks.")

    def search(self, query: str, k: int = 4):
        if self.index is None or not self.chunks:
            return []
        q = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(q)
        scores, idxs = self.index.search(q, k)
        results = []
        for score, idx in zip(scores[0], idxs[0]):
            if idx == -1: continue
            results.append((float(score), self.chunks[idx]))
        return results

rag = RAGIndex(EMBEDDING_MODEL)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [16]:
def render_context(snippets):
    lines = []
    for rank, (score, ch) in enumerate(snippets, start=1):
        header = f"[Chunk {rank}] (score={score:.3f}) source={os.path.basename(ch['source'])} id={ch['chunk_id']}"
        lines.append(header + "\n" + ch["text"])
    return "\n\n".join(lines)

def build_prompt(question, context_blocks):
    return ANSWER_TEMPLATE.format(
        system=SYSTEM_PROMPT.strip(),
        context=context_blocks.strip(),
        question=question.strip()
    )

In [17]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import torch

_local_pipe = None

def get_local_pipe():
    global _local_pipe
    if _local_pipe is None:
        tok = AutoTokenizer.from_pretrained(LOCAL_MODEL, use_fast=True)
        model = AutoModelForCausalLM.from_pretrained(
            LOCAL_MODEL,
            device_map="auto" if torch.cuda.is_available() else None,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            low_cpu_mem_usage=True
        )
        _local_pipe = pipeline(
            "text-generation",
            model=model,
            tokenizer=tok,
            device=0 if torch.cuda.is_available() else -1,
            max_new_tokens=384,
            do_sample=True,
            temperature=0.3,
            top_p=0.9,
            repetition_penalty=1.05
        )
    return _local_pipe

def generate_local(prompt: str) -> str:
    p = get_local_pipe()
    out = p(prompt, pad_token_id=p.tokenizer.eos_token_id)[0]["generated_text"]
    return out[len(prompt):].strip()

In [18]:
def generate_openai(prompt: str) -> str:
    if not OPENAI_API_KEY:
        return "OPENAI_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."
    try:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        r = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role":"system","content":SYSTEM_PROMPT},
                      {"role":"user","content":prompt}],
            temperature=0.2,
            max_tokens=400
        )
        return r.choices[0].message.content
    except Exception as e:
        return f"[OpenAI error] {e}"

In [19]:
def _strip_local_proxy_env():
    # VS Code Colab plugin may inject localhost proxy envs that timeout.
    keys = [
        "HTTP_PROXY", "HTTPS_PROXY", "ALL_PROXY",
        "http_proxy", "https_proxy", "all_proxy",
    ]
    for k in keys:
        v = os.environ.get(k, "")
        if "localhost:" in v or "127.0.0.1:" in v:
            os.environ.pop(k, None)


def _extract_gemini_text(resp_json: dict) -> str:
    cands = resp_json.get("candidates", [])
    if not cands:
        return ""
    parts = cands[0].get("content", {}).get("parts", [])
    texts = [p.get("text", "") for p in parts if isinstance(p, dict)]
    return "\n".join([t for t in texts if t]).strip()


def _generate_gemini_via_sdk(prompt: str, model_name: str = "gemini-2.5-flash-lite", timeout_sec: int = 35) -> str:
    import google.generativeai as genai

    genai.configure(api_key=GOOGLE_API_KEY)
    model = genai.GenerativeModel(model_name)
    r = model.generate_content(prompt, request_options={"timeout": timeout_sec})
    return (getattr(r, "text", "") or "").strip()


def _generate_gemini_via_rest(prompt: str, model_name: str = "gemini-2.5-flash-lite", timeout_sec: int = 35) -> str:
    import requests

    url = f"https://generativelanguage.googleapis.com/v1beta/models/{model_name}:generateContent"
    payload = {"contents": [{"parts": [{"text": prompt}]}]}

    # Explicitly disable proxies for this request.
    r = requests.post(
        url,
        params={"key": GOOGLE_API_KEY},
        json=payload,
        timeout=timeout_sec,
        proxies={"http": "", "https": ""},
    )
    r.raise_for_status()
    data = r.json()
    text = _extract_gemini_text(data)
    if text:
        return text
    return f"[Gemini REST response without text] {str(data)[:400]}"


def generate_gemini(prompt: str) -> str:
    if not GOOGLE_API_KEY:
        return "GOOGLE_API_KEY not set. Switch GEN_BACKEND to 'local' or set your key."

    _strip_local_proxy_env()

    try:
        return _generate_gemini_via_sdk(prompt)
    except Exception as sdk_err:
        try:
            return _generate_gemini_via_rest(prompt)
        except Exception as rest_err:
            return f"[Gemini error] sdk={sdk_err} | rest={rest_err}"


In [20]:
def answer_question(question: str, top_k: int = TOP_K):
    hits = rag.search(question, k=top_k)
    context = render_context(hits)
    prompt = build_prompt(question, context)

    if GEN_BACKEND == "local":
        answer = generate_local(prompt)
    elif GEN_BACKEND == "openai":
        answer = generate_openai(prompt)
    elif GEN_BACKEND == "gemini":
        answer = generate_gemini(prompt)
    else:
        answer = f"Unknown backend: {GEN_BACKEND}"

    return {"question": question, "answer": answer, "top_chunks": hits}

print("RAG ready. After indexing, call: answer_question('Your query')")

RAG ready. After indexing, call: answer_question('Your query')


In [21]:
import os
from typing import List

SUPPORTED_INDEX_EXTS = (".pdf", ".txt", ".md", ".docx", ".doc")


def _collect_supported_files(scan_dir: str) -> List[str]:
    paths = []
    if not scan_dir or not os.path.isdir(scan_dir):
        return paths

    for root, _, files in os.walk(scan_dir):
        for name in files:
            if name.lower().endswith(SUPPORTED_INDEX_EXTS):
                paths.append(os.path.join(root, name))
    return sorted(paths)


def _upload_files_widget() -> List[str]:
    """Works only in active Chrome Colab browser sessions."""
    try:
        from google.colab import files
    except Exception as e:
        print(f"Colab upload widget unavailable: {e}")
        return []

    try:
        print("Select PDFs/TXT/MD/DOCX/DOC files...")
        uploaded = files.upload()
    except Exception as e:
        print(f"Upload widget failed: {e}")
        return []

    paths = []
    target_dir = "/content" if os.path.isdir("/content") else os.getcwd()
    for name, data in uploaded.items():
        out_path = os.path.join(target_dir, name)
        with open(out_path, "wb") as f:
            f.write(data)
        paths.append(out_path)
    return paths


def resolve_index_paths(
    manual_paths: List[str] = None,
    scan_dir: str = "",
    use_widget_upload: bool = False,
) -> List[str]:
    # 1) Highest priority: explicit manual paths
    manual_paths = manual_paths or []
    existing = [p for p in manual_paths if isinstance(p, str) and os.path.isfile(p)]
    if existing:
        return existing

    # 2) Optional browser widget upload (Chrome Colab only)
    if use_widget_upload:
        widget_paths = _upload_files_widget()
        if widget_paths:
            return widget_paths
        print("Widget upload returned no files. Falling back to directory scan...")

    # 3) Directory scan fallback
    scanned = _collect_supported_files(scan_dir)
    if scanned:
        print(f"Found {len(scanned)} supported files in: {scan_dir}")
    return scanned


def build_index_from_paths(paths: List[str]):
    if not paths:
        raise ValueError("No input files found. Set MANUAL_PATHS or SCAN_DIR first.")

    docs = load_texts_from_paths(paths)
    chunks = chunk_docs(docs)
    rag.build(chunks)
    print(f"Indexed {len(chunks)} chunks from {len(docs)} files.")


### Optional: Build / Rebuild RAG Index

Use this only when you want to index files for Impact Agent retrieval.


In [22]:
# Optional index build command (manual)
# Example:
# paths = resolve_index_paths(manual_paths=["/content/drive/MyDrive/your_script.pdf"], scan_dir="/content/drive/MyDrive", use_widget_upload=False)
# build_index_from_paths(paths)
# print("Chunks indexed:", len(rag.chunks))


## STEP 14: Unified UI (V4 Style + Orchestrator Dispatch)

- UI style copied from `Impact_RAG_v4`.
- Uses `safe_route` from orchestrator module each turn.
- Dispatches to:
  - Impact Agent (`_gradio_story_eval`)
  - Script Reviewer (`evaluate_story`)

Note: This cell defines `demo` but does not launch it.


In [23]:
import os
import gradio as gr
import inspect
from types import SimpleNamespace
from pypdf import PdfReader
from docx import Document


def _extract_uploaded_path(uploaded_file) -> str:
    if uploaded_file is None:
        return ""
    if isinstance(uploaded_file, str):
        return uploaded_file
    if isinstance(uploaded_file, list) and uploaded_file:
        return _extract_uploaded_path(uploaded_file[0])
    if isinstance(uploaded_file, dict):
        return uploaded_file.get("name", "") or uploaded_file.get("path", "")
    if hasattr(uploaded_file, "name"):
        return uploaded_file.name
    return ""


def read_file_to_string(file_path: str) -> str:
    """Reads a TXT, PDF, DOCX, or DOC file into a single text string."""
    if not file_path:
        return ""

    text = ""
    file_ext = os.path.splitext(file_path)[1].lower()

    try:
        if file_ext == ".pdf":
            reader = PdfReader(file_path)
            for page in reader.pages:
                text += page.extract_text() or ""

        elif file_ext == ".docx":
            doc = Document(file_path)
            text = "\n".join([p.text for p in doc.paragraphs])
            for table in doc.tables:
                for row in table.rows:
                    for cell in row.cells:
                        text += "\n" + cell.text

        elif file_ext == ".doc":
            try:
                doc = Document(file_path)
                text = "\n".join([p.text for p in doc.paragraphs])
            except Exception:
                return "ERROR: Legacy .doc detected. Please resave as .docx and upload again."

        elif file_ext in [".txt", ".md"]:
            with open(file_path, "r", encoding="utf-8", errors="replace") as f:
                text = f.read()

        else:
            return f"ERROR: Unsupported file type: {file_ext}"

    except Exception as e:
        return f"ERROR: Could not read file ({e})"

    return text


def on_upload(uploaded_file):
    file_path = _extract_uploaded_path(uploaded_file)
    if not file_path:
        return "", "No file attached.", ""

    file_text = read_file_to_string(file_path)
    if file_text.startswith("ERROR:"):
        return "", file_text, ""

    filename = os.path.basename(file_path)
    return file_text, f"Attached: {filename}", file_path


def clear_uploaded_state():
    return "", "No file attached.", ""


def _combine_user_input(story_text: str, uploaded_text: str) -> str:
    story = (story_text or "").strip()
    file_body = (uploaded_text or "").strip()

    if story and file_body:
        return f"{story}\n\n[Uploaded File Content]\n{file_body}"
    return story or file_body


def _format_user_turn(story_text: str, uploaded_text: str) -> str:
    story = (story_text or "").strip()
    has_file = bool((uploaded_text or "").strip())

    if story and has_file:
        return story + "\n[Attachment included]"
    if story:
        return story
    if has_file:
        return "[Attachment only submission]"
    return ""


def _gradio_story_eval(story_text: str, mission: str = ""):
    """Impact agent path (copied from v4)."""
    if not story_text.strip():
        return "Please enter text or upload a file before submitting."

    base_prompt = """
    You are an editorial advisor focused on community engagement and ethical publication. Read the following piece of writing
    and identify the communities, audiences, or individuals who may be directly affected by its themes. Then provide specific,
    actionable suggestions for how the author can responsibly and meaningfully reach out to or support those communities.

    Your feedback must:
    - Be concrete (exact additions, placements, wording ideas, or resources).
    - Explain why each suggestion is appropriate for the content.
    - Address ethical considerations such as harm reduction, accessibility, and care for vulnerable readers when relevant.

    Avoid generic advice like "be sensitive" or "raise awareness."

    When users share a script, concept, or idea, your job is to:
    1. Analyze the submission's tone, themes, and emotional depth.
    2. Determine the topics discussed in the submission.
    3. Provide clear reasoning.
    4. Provide specific and actionable outreach/support suggestions.
    5. Integrate any mission context naturally.

    Respond in this format:

    Topics identified:
    (List topics found in the submission. For each topic include one quote and one actionable recommendation.)
    """

    if mission and mission.strip():
        user_prompt = (
            f"{base_prompt}\n\nCurrent Studio Mission:\n{mission.strip()}\n\nUser Submission:\n{story_text.strip()}"
        )
    else:
        user_prompt = f"{base_prompt}\n\nUser Submission:\n{story_text.strip()}"

    context_text = ""
    if "rag" in globals() and getattr(rag, "chunks", []):
        hits = rag.search(story_text, k=TOP_K)
        context_text = render_context(hits)
    else:
        hits = []

    full_prompt = ANSWER_TEMPLATE.format(
        system=SYSTEM_PROMPT,
        context=context_text or "[No extra context]",
        question=user_prompt,
    )

    out = answer_question(full_prompt)
    answer = out.get("answer", "")

    cites = []
    if context_text:
        for rank, (_, ch) in enumerate(out.get("top_chunks", []), start=1):
            cites.append(f"Chunk {rank} - {os.path.basename(ch['source'])}#{ch['chunk_id']}")
    suffix = ("\n\n---\nSources: " + "; ".join(cites)) if cites else ""

    return (answer or "[Empty answer]") + suffix


def _to_uploaded_file_obj(path: str):
    if not path:
        return None
    return SimpleNamespace(name=path)


def _dispatch_request(story_text: str, mission_text: str, uploaded_text: str, uploaded_path: str):
    if "safe_route" not in globals():
        return "error", "missing_safe_route", "Routing module not initialized. Run orchestrator cells first."

    merged_submission = _combine_user_input(story_text, uploaded_text)
    route_basis = (story_text or "").strip() or merged_submission[:1200]
    if not route_basis:
        return "error", "empty_input", "Please enter text or upload a file before submitting."

    routed_agent, route_mode = safe_route(route_basis)

    if routed_agent == orchestrator.IMPACT_AGENT:
        answer_body = _gradio_story_eval(merged_submission, mission_text or "")
    else:
        file_obj = _to_uploaded_file_obj(uploaded_path)
        # Keep script-reviewer behavior aligned with original notebook: text input + optional file.
        script_input = (story_text or "").strip()
        answer_body, _, _ = evaluate_story(script_input, mission_text or "", file_obj)

    final_answer = f"**Routed Agent:** `{routed_agent}`\n\n**Route Mode:** `{route_mode}`\n\n{answer_body}"
    return routed_agent, route_mode, final_answer


def stage_user_message(story_text: str, mission_text: str, uploaded_text: str, uploaded_path: str, chat_history):
    merged_submission = _combine_user_input(story_text, uploaded_text)
    if not merged_submission.strip():
        history = list(chat_history or [])
        return history, "", history, {}

    history = list(chat_history or [])
    history.append({"role": "user", "content": _format_user_turn(story_text, uploaded_text)})
    history.append({"role": "assistant", "content": "Analyzing your submission..."})

    payload = {
        "story_text": story_text or "",
        "mission_text": mission_text or "",
        "uploaded_text": uploaded_text or "",
        "uploaded_path": uploaded_path or "",
    }

    return history, "", history, payload


def resolve_assistant_message(chat_history, pending_payload):
    history = list(chat_history or [])
    payload = pending_payload or {}

    if not payload:
        return history, history, {}

    routed_agent, route_mode, response_text = _dispatch_request(
        payload.get("story_text", ""),
        payload.get("mission_text", ""),
        payload.get("uploaded_text", ""),
        payload.get("uploaded_path", ""),
    )

    if history and history[-1].get("role") == "assistant":
        history[-1] = {"role": "assistant", "content": response_text}
    else:
        history.append({"role": "assistant", "content": response_text})

    return history, history, {}


def clear_chat_state():
    return [], "", [], {}, "", "", "No file attached."


CUSTOM_CSS = """
@import url('https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&display=swap');

:root {
    --bg-page: #cae0ee;
    --bg-shell: #eaf4fb;
    --bg-side: #d8e7f2;
    --bg-chat: linear-gradient(180deg, #0a1630 0%, #0b1f3f 100%);
    --bg-bot: rgba(14, 35, 67, 0.94);
    --bg-user: rgba(71, 87, 113, 0.9);
    --bg-composer: #f8fbfe;
    --text-main: #10233c;
    --text-muted: #4e637f;
    --text-light: #eaf2ff;
    --border-soft: rgba(136, 161, 194, 0.35);
    --border-input: #c7d8e8;
    --brand: #0a5cad;
    --brand-2: #0c74d8;
}

.gradio-container {
    background: radial-gradient(circle at 20% 0%, #d8ecf8 0%, var(--bg-page) 52%, #bfd8e8 100%) !important;
    font-family: 'Space Grotesk', 'Avenir Next', sans-serif !important;
    color: var(--text-main) !important;
}

#main-layout {
    max-width: 1260px;
    margin: 0 auto;
    padding: 20px;
    gap: 14px;
    min-height: 100vh;
}

.app-sidebar {
    background: linear-gradient(180deg, #dcecf7 0%, var(--bg-side) 100%) !important;
    border: 1px solid var(--border-soft) !important;
    border-radius: 18px !important;
    padding: 16px !important;
    box-shadow: 0 12px 30px rgba(31, 52, 80, 0.12);
}

.brand-pill {
    display: inline-block;
    background: #f4f9ff;
    color: var(--brand);
    font-weight: 700;
    font-size: 28px;
    letter-spacing: 0.3px;
    border: 1px solid #d6e6f5;
    border-radius: 12px;
    padding: 10px 14px;
    margin-bottom: 14px;
}

.sidebar-btn button {
    width: 100%;
    background: #4b586b !important;
    color: #f8fbff !important;
    border: 1px solid rgba(255, 255, 255, 0.12) !important;
    border-radius: 10px !important;
    height: 44px;
    font-weight: 600;
    margin-bottom: 8px;
}

.sidebar-btn button:hover {
    background: #3f4a5a !important;
}

.mission-wrap {
    margin-top: 8px;
    background: #1a2940;
    border-radius: 12px;
    padding: 10px;
    border: 1px solid rgba(255, 255, 255, 0.14);
}

.mission-wrap label {
    color: #9fc4ff !important;
    font-weight: 600 !important;
}

.mission-wrap textarea {
    background: #f7fbff !important;
    color: #10233c !important;
    border: 1px solid #bfd4ea !important;
    border-radius: 9px !important;
}

.mission-wrap textarea::placeholder {
    color: #6e8098 !important;
}

.main-shell {
    background: var(--bg-shell) !important;
    border: 1px solid var(--border-soft) !important;
    border-radius: 20px !important;
    padding: 0 !important;
    display: flex !important;
    flex-direction: column !important;
    box-shadow: 0 16px 36px rgba(24, 45, 72, 0.14);
}

.main-header {
    padding: 22px 24px 12px 24px;
    border-bottom: 1px solid rgba(129, 154, 185, 0.28);
}

.main-header h1 {
    margin: 0;
    color: var(--brand);
    font-size: 42px;
    letter-spacing: 0.2px;
}

.main-header p {
    margin: 6px 0 0 0;
    color: var(--text-muted);
    font-size: 15px;
}

.chat-surface {
    padding: 14px;
}

.chat-surface > .column {
    background: var(--bg-chat) !important;
    border-radius: 14px !important;
    border: 1px solid rgba(141, 169, 203, 0.28) !important;
    min-height: 56vh;
    max-height: 64vh;
    overflow: auto !important;
    padding: 14px !important;
}

.chat-stream {
    background: transparent !important;
}

.chat-stream .message {
    border-radius: 14px !important;
    border: 1px solid rgba(154, 181, 213, 0.24) !important;
    padding: 12px 14px !important;
    line-height: 1.6 !important;
}

.chat-stream .message p,
.chat-stream .message li,
.chat-stream .message span,
.chat-stream .message div {
    color: inherit !important;
}

.chat-stream .message.user {
    background: var(--bg-user) !important;
    color: #eef5ff !important;
}

.chat-stream .message.bot,
.chat-stream .message.assistant {
    background: var(--bg-bot) !important;
    color: var(--text-light) !important;
}

.composer-wrap {
    border-top: 1px solid rgba(129, 154, 185, 0.25);
    padding: 14px;
    background: var(--bg-composer);
}

.composer-row {
    border: 1px solid var(--border-input);
    border-radius: 16px;
    padding: 8px;
    background: #ffffff;
    align-items: center !important;
    gap: 8px !important;
}

.attach-btn button {
    min-width: 116px;
    height: 44px;
    border-radius: 10px !important;
    border: 1px solid #d2e0ee !important;
    background: #4b586b !important;
    color: #f3f8ff !important;
    font-weight: 600;
}

.attach-btn button:hover {
    background: #3f4a5a !important;
}

.composer-input,
.composer-input .wrap,
.composer-input .block,
.composer-input .container {
    background: transparent !important;
}

.composer-input,
.composer-input * {
    color: #10233c !important;
}

.composer-input textarea {
    border: none !important;
    box-shadow: none !important;
    background: transparent !important;
    color: #10233c !important;
    caret-color: #10233c !important;
    font-size: 17px !important;
    font-weight: 500 !important;
}

.composer-input textarea::placeholder {
    color: #6e8098 !important;
}

.send-btn button {
    min-width: 112px;
    height: 44px;
    border-radius: 10px !important;
    border: none !important;
    background: linear-gradient(135deg, var(--brand) 0%, var(--brand-2) 100%) !important;
    color: #f7fbff !important;
    font-weight: 700;
}

.send-btn button:hover {
    filter: brightness(1.04);
}

.file-status,
.file-status p,
.file-status span {
    margin: 6px 0 0 2px !important;
    color: #2f4765 !important;
    font-size: 16px !important;
    font-weight: 700 !important;
    font-style: italic !important;
    letter-spacing: 0.1px;
    opacity: 1 !important;
    word-break: break-word;
}

.clear-file-btn button {
    margin-top: 8px;
    border-radius: 10px !important;
    border: 1px solid #d2e0ee !important;
    background: #ffffff !important;
    color: #364e6d !important;
    height: 40px;
}

[data-testid="status-bar"],
.progress-text,
.meta-text,
.eta-bar,
.pending {
    display: none !important;
}

@media (max-width: 980px) {
    #main-layout {
        padding: 10px;
        gap: 10px;
    }

    .brand-pill {
        font-size: 22px;
    }

    .main-header h1 {
        font-size: 34px;
    }

    .chat-surface > .column {
        min-height: 52vh;
        max-height: 58vh;
    }
}
"""


_gradio_version = getattr(gr, "__version__", "0")
try:
    _gradio_major = int(str(_gradio_version).split(".")[0])
except Exception:
    _gradio_major = 0

if _gradio_major >= 6:
    # Gradio 6 moved css/theme to launch(); keep constructor compatible.
    _blocks_kwargs = {"fill_height": True}
    UI_LAUNCH_STYLE_KW = {"css": CUSTOM_CSS, "theme": gr.themes.Base()}
else:
    _blocks_kwargs = {"css": CUSTOM_CSS, "theme": gr.themes.Base(), "fill_height": True}
    UI_LAUNCH_STYLE_KW = {}

with gr.Blocks(**_blocks_kwargs) as demo:
    uploaded_text_state = gr.State("")
    uploaded_path_state = gr.State("")
    chat_state = gr.State([])
    pending_payload_state = gr.State({})

    with gr.Row(elem_id="main-layout"):
        with gr.Column(scale=0, min_width=260, elem_classes="app-sidebar"):
            gr.HTML('<div class="brand-pill">Impact Studios</div>')

            new_chat_btn = gr.Button("New chat", elem_classes="sidebar-btn")
            search_btn = gr.Button("Search chat", elem_classes="sidebar-btn")
            add_agent_btn = gr.Button("Add agent", elem_classes="sidebar-btn")

            with gr.Column(elem_classes="mission-wrap"):
                mission_box = gr.Textbox(
                    label="Studio Mission (optional)",
                    placeholder="Add mission context for this run...",
                    lines=6,
                )

        with gr.Column(scale=1, elem_classes="main-shell"):
            gr.HTML(
                """
                <div class="main-header">
                    <h1>Impact Studios</h1>
                    <p>Unified: Orchestrator + Script Reviewer + Impact Agent</p>
                </div>
                """
            )

            with gr.Column(elem_classes="chat-surface"):
                _chatbot_sig = set(inspect.signature(gr.Chatbot.__init__).parameters.keys())
                _chatbot_kwargs = {
                    "value": [],
                    "show_label": False,
                    "container": False,
                    "bubble_full_width": False,
                    "elem_classes": "chat-stream",
                }
                if "type" in _chatbot_sig:
                    _chatbot_kwargs["type"] = "messages"
                elif "message_type" in _chatbot_sig:
                    _chatbot_kwargs["message_type"] = "messages"

                chatbot = gr.Chatbot(**{k: v for k, v in _chatbot_kwargs.items() if k in _chatbot_sig})

            with gr.Column(elem_classes="composer-wrap"):
                with gr.Row(elem_classes="composer-row", equal_height=True):
                    upload_btn = gr.UploadButton(
                        "Attach",
                        file_types=[".txt", ".pdf", ".docx", ".doc", ".md"],
                        file_count="single",
                        elem_classes="attach-btn",
                        scale=0,
                    )

                    story = gr.Textbox(
                        label="",
                        placeholder="Ask anything",
                        lines=1,
                        max_lines=5,
                        show_label=False,
                        container=False,
                        elem_classes="composer-input",
                        scale=1,
                    )

                    submit_btn = gr.Button("Send", elem_classes="send-btn", scale=0)

                file_status = gr.Markdown("No file attached.", elem_classes="file-status")
                clear_file_btn = gr.Button("Clear attached file", elem_classes="clear-file-btn")

    upload_btn.upload(
        fn=on_upload,
        inputs=upload_btn,
        outputs=[uploaded_text_state, file_status, uploaded_path_state],
        show_progress="hidden",
    )

    clear_file_btn.click(
        fn=clear_uploaded_state,
        inputs=None,
        outputs=[uploaded_text_state, file_status, uploaded_path_state],
        show_progress="hidden",
    )

    submit_event = submit_btn.click(
        fn=stage_user_message,
        inputs=[story, mission_box, uploaded_text_state, uploaded_path_state, chat_state],
        outputs=[chatbot, story, chat_state, pending_payload_state],
        show_progress="hidden",
    )

    submit_event.then(
        fn=resolve_assistant_message,
        inputs=[chat_state, pending_payload_state],
        outputs=[chatbot, chat_state, pending_payload_state],
        show_progress="hidden",
    )

    enter_event = story.submit(
        fn=stage_user_message,
        inputs=[story, mission_box, uploaded_text_state, uploaded_path_state, chat_state],
        outputs=[chatbot, story, chat_state, pending_payload_state],
        show_progress="hidden",
    )

    enter_event.then(
        fn=resolve_assistant_message,
        inputs=[chat_state, pending_payload_state],
        outputs=[chatbot, chat_state, pending_payload_state],
        show_progress="hidden",
    )

    new_chat_btn.click(
        fn=clear_chat_state,
        inputs=None,
        outputs=[
            chatbot,
            story,
            chat_state,
            pending_payload_state,
            uploaded_text_state,
            uploaded_path_state,
            file_status,
        ],
        show_progress="hidden",
    )

print("[STEP 14] Unified UI object is ready as variable: demo")


[STEP 14] Unified UI object is ready as variable: demo


### Launch UI

Run the next cell to launch Gradio UI.


In [24]:
# STEP 14.1: Launch Gradio UI
if "demo" not in globals():
    raise RuntimeError("demo is not defined. Please run STEP 14 first.")

launch_kwargs = {"share": True, "debug": True}
if isinstance(globals().get("UI_LAUNCH_STYLE_KW"), dict):
    launch_kwargs.update(UI_LAUNCH_STYLE_KW)

print("[STEP 14.1] Launching Gradio with kwargs:", launch_kwargs)
demo.launch(**launch_kwargs)



[STEP 14.1] Launching Gradio with kwargs: {'share': True, 'debug': True, 'css': '\n@import url(\'https://fonts.googleapis.com/css2?family=Space+Grotesk:wght@400;500;600;700&display=swap\');\n\n:root {\n    --bg-page: #cae0ee;\n    --bg-shell: #eaf4fb;\n    --bg-side: #d8e7f2;\n    --bg-chat: linear-gradient(180deg, #0a1630 0%, #0b1f3f 100%);\n    --bg-bot: rgba(14, 35, 67, 0.94);\n    --bg-user: rgba(71, 87, 113, 0.9);\n    --bg-composer: #f8fbfe;\n    --text-main: #10233c;\n    --text-muted: #4e637f;\n    --text-light: #eaf2ff;\n    --border-soft: rgba(136, 161, 194, 0.35);\n    --border-input: #c7d8e8;\n    --brand: #0a5cad;\n    --brand-2: #0c74d8;\n}\n\n.gradio-container {\n    background: radial-gradient(circle at 20% 0%, #d8ecf8 0%, var(--bg-page) 52%, #bfd8e8 100%) !important;\n    font-family: \'Space Grotesk\', \'Avenir Next\', sans-serif !important;\n    color: var(--text-main) !important;\n}\n\n#main-layout {\n    max-width: 1260px;\n    margin: 0 auto;\n    padding: 20px;\n

Unsloth: Input IDs of shape torch.Size([1, 9328]) with length 9328 > the model's max sequence length of 2048.
We shall truncate it ourselves. It's imperative if you correct this issue first.



[DEBUG] FINAL STORY (user prompt + uploaded script text)
Please review my script and tell me what to improve.

 
Treatment  
The  young  man  James  is  in  his  mid  twenties,  handsome.  He  is  already  awake,  tangled  in  his                  
sheets  and  lingering  in  bed.  He  gets  up  and  takes  a  freezing  cold  shower,  gasping  but  powering                  
through.  
He  commutes  to  work,  first  walking  the  streets  with  a  slow,  deliberate  purpose  in  the  mild  cold.                  
He  quietly  and  detachedly  observes  as  things  happen  around  him.  He  notices  other  people,               
their  everyday  interactions,  their  conflicts.  He  hurries  into  the  subway,  narrowly  missing  a  train               
but  remains  unphased.  He  steps  onto  the  train:  it’s  busy.  A  woman  is  subtly  groped  in  the                  
subway  by  a  creep.  “Back  off”  she  says,  she  and  James’  eyes  meet  briefly,  understanding          

--- Logging error ---
Traceback (most recent call last):
  File "/usr/lib/python3.12/logging/__init__.py", line 1160, in emit
    msg = self.format(record)
          ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 999, in format
    return fmt.format(record)
           ^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 703, in format
    record.message = record.getMessage()
                     ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/logging/__init__.py", line 392, in getMessage
    msg = msg % self.args
          ~~~~^~~~~~~~~~~
TypeError: not all arguments converted during string formatting
Call stack:
  File "/usr/lib/python3.12/threading.py", line 1032, in _bootstrap
    self._bootstrap_inner()
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    self.run()
  File "/usr/local/lib/python3.12/dist-packages/anyio/_backends/_asyncio.py", line 986, in run
    result = context.run(func, *args)
  File "/usr


[DEBUG] FINAL STORY (user prompt + uploaded script text)
Please review my script and tell me what to improve.

A coalition of teachers, engineers, and local volunteers launches a free community learning center that offers open-access STEM workshops, AI literacy classes, and mentorship for students from underserved backgrounds.
The program runs year-round, provides laptops and materials at no cost, and pairs each student with a trained mentor who supports both academic growth and emotional well-being. Alumni return to become new mentors, creating a self-sustaining cycle of empowerment.
The center is designed to be inclusive for learners of all ages and abilities, and all curriculum materials are open-source so that other communities can replicate the model.
Evaluate whether this initiative uplifts humanity, provide a score from 0 to 1, explain benefits and risks, and give a concise recommendation for scaling.



[DEBUG] FULL PROMPT SENT TO MODEL
Below is an instruction that describes a